In [11]:
using Oceananigans, CairoMakie
using Oceananigans.OutputWriters
using NCDatasets

include("../jets/utils/parse_config.jl")

parse_config (generic function with 1 method)

In [12]:
EXPERIMENT = "atm_norm"
config_path = "../config/$(EXPERIMENT).yaml"
output_path = "../data/processed/$(EXPERIMENT).nc"

"../data/processed/atm_norm.nc"

In [13]:
@info "Parsing configuration: $config_path"
params = parse_config(config_path)
@info "Configuration parsed" params

[ Info: Parsing configuration: ../config/atm_norm.yaml
┌ Info: Configuration parsed
│   params =
│    (arch = CPU(), precision = Float64, n_grid = 128, advection = WENO{3, Float64, Nothing}(order=5)
│    ├── buffer_scheme: WENO{2, Float64, Nothing}(order=3)
│    │   └── buffer_scheme: Centered(order=2)
└    └── advecting_velocity_scheme: Centered(order=4), timestepper = :RungeKutta3, Δt = 0.01, stop_iteration = 2000)


In [14]:
@info "Initializing grid"
grid = RectilinearGrid(
    params.arch,
    params.precision;
    size = (params.n_grid, params.n_grid),
    x = (0, 2π),
    y = (0, 2π),
    topology = (Periodic, Periodic, Flat),
)
@info "Grid initialized" grid

[ Info: Initializing grid
┌ Info: Grid initialized
│   grid =
│    128×128×1 RectilinearGrid{Float64, Periodic, Periodic, Flat} on CPU with 3×3×0 halo
│    ├── Periodic x ∈ [-7.51279e-18, 6.28319) regularly spaced with Δx=0.0490874
│    ├── Periodic y ∈ [-7.51279e-18, 6.28319) regularly spaced with Δy=0.0490874
└    └── Flat z                               


In [15]:
@info "Initializing model"
model = NonhydrostaticModel(
    grid;
    advection = params.advection,
    timestepper = params.timestepper,
)
@info "Model initialized"

[ Info: Initializing model
[ Info: Model initialized


In [16]:
@info "Initializing model state"
ϵ(x, y) = 2 * rand() - 1
set!(
    model;
    u = ϵ,
    v = ϵ,
)

[ Info: Initializing model state


In [17]:
@info "Initializing diagnostics"
ζ = Field(
    ∂x(model.velocities.v) -
    ∂y(model.velocities.u)
)
outputs = Dict(
    "u" => model.velocities.u,
    "v" => model.velocities.v,
    "ζ" => ζ,
)

[ Info: Initializing diagnostics


Dict{String, Field{LX, LY, Center, O, RectilinearGrid{Float64, Periodic, Periodic, Flat, Oceananigans.Grids.StaticVerticalDiscretization{Nothing, Nothing, Float64, Float64}, Float64, Float64, OffsetArrays.OffsetVector{Float64, StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}}, OffsetArrays.OffsetVector{Float64, StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}}, CPU, Oceananigans.Grids.GridSize{128, 128, 1, 3, 3, 0}}, Tuple{Colon, Colon, Colon}, OffsetArrays.OffsetArray{Float64, 3, Array{Float64, 3}}, Float64, FieldBoundaryConditions{BoundaryCondition{Oceananigans.BoundaryConditions.Periodic, Nothing}, BoundaryCondition{Oceananigans.BoundaryConditions.Periodic, Nothing}, BoundaryCondition{Oceananigans.BoundaryConditions.Periodic, Nothing}, BoundaryCondition{Oceananigans.BoundaryConditions.Periodic, Nothing}, Nothing, Nothing, Nothing, @NamedTuple{bottom_and_top::Nothing, south_and_north::Oceananigans.Boundar

In [18]:
@info "Initializing NetCDF writer"
nc_writer = NetCDFWriter(
    model,
    outputs;
    filename = output_path,
    schedule = IterationInterval(100),
    overwrite_existing = true,
)

[ Info: Initializing NetCDF writer


NetCDFWriter scheduled on IterationInterval(100):
├── filepath: ../data/processed/atm_norm.nc
├── dimensions: time(0), y_afa(128), x_faa(128), x_caa(128), y_aca(128)
├── 3 outputs: (v, u, ζ)
├── array_type: Array{Float32}
├── file_splitting: NoFileSplitting
└── file size: 29.9 KiB

In [19]:
@info "Initializing simulation"
simulation = Simulation(
    model;
    Δt = params.Δt,
    stop_iteration = params.stop_iteration,
)
simulation.output_writers[:nc] = nc_writer

[ Info: Initializing simulation


NetCDFWriter scheduled on IterationInterval(100):
├── filepath: ../data/processed/atm_norm.nc
├── dimensions: time(0), y_afa(128), x_faa(128), x_caa(128), y_aca(128)
├── 3 outputs: (v, u, ζ)
├── array_type: Array{Float32}
├── file_splitting: NoFileSplitting
└── file size: 29.9 KiB

In [20]:
show_time(sim) =
    "Time is $(prettytime(sim.model.clock.time))"

simulation.callbacks[:show_time] =
    Callback(show_time, IterationInterval(100))

Callback of show_time on IterationInterval(100)

In [21]:
simulation.stop_iteration += 0
@info "Starting simulation"
run!(simulation)
@info "Simulation finished"

[ Info: Starting simulation
[ Info: Initializing simulation...
[ Info:     ... simulation initialization complete (1.552 seconds)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (1.416 seconds).
[ Info: Simulation is stopping after running for 19.776 seconds.
[ Info: Model iteration 2000 equals or exceeds stop iteration 2000.
[ Info: Simulation finished


In [48]:
simulation.stop_iteration += 0
run!(simulation)
@info "Simulation finished"

[ Info: Initializing simulation...
[ Info:     ... simulation initialization complete (4.355 ms)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (16.360 ms).
[ Info: Simulation is stopping after running for 6.736 seconds.
[ Info: Model iteration 100000 equals or exceeds stop iteration 100000.
[ Info: Simulation finished
